# OpenFold3 Quickstart

Install [OpenFold3](https://github.com/aqlaboratory/openfold-3), predict a structure, and train
on one GPU. Every command below is the command you type: the notebook runs them as shell cells
and keeps the real output.

| | |
|---|---|
| 1. Install | pixi environment, model parameters |
| 2. Inference | predict ubiquitin from sequence |
| 3. Training | two epochs on a small PDB subset |
| 4. Scaling | what changes for multi-GPU and the full dataset |

> Executed on a single [NVIDIA H100 80GB HBM3](https://www.nvidia.com/en-us/data-center/h100/).
> Inference took about four minutes, training about five, and peak GPU memory was 28 GiB.

## 1. Install

Clone the repo and build the CUDA environment. This part is a shell prerequisite; run it
before starting the notebook:

```bash
git clone https://github.com/aqlaboratory/openfold-3.git
cd openfold-3
pixi install -e openfold3-cuda12
```

The environment is about 14 GB and needs [pixi](https://pixi.sh) 0.80+. Everything after this
runs through `pixi run -e openfold3-cuda12`.

Point the notebook at your checkout. Every cell below runs from there.

In [1]:
# EDIT THIS: your openfold-3 checkout.
%cd /home/ubuntu/jianwen/openfold-3

/home/ubuntu/jianwen/openfold-3


In [2]:
!pixi run -e openfold3-cuda12 python -c "\
import torch; from importlib.metadata import version; \
c = torch.cuda.get_device_capability(0); \
print('openfold3 ', version('openfold3')); \
print('torch     ', torch.__version__, '| CUDA', torch.version.cuda); \
print('GPU       ', torch.cuda.get_device_name(0), f'(sm_{c[0]}{c[1]})')" 2>/dev/null

openfold3  0.5.1.dev55+g3a9ff907f
torch      2.10.0 | CUDA 12.9
GPU        NVIDIA H100 80GB HBM3 (sm_90)


### Model parameters

Inference needs trained weights. `setup_openfold` downloads them to `~/.openfold3/` (about
4.3 GB) and fetches the chemical component dictionary. It is idempotent: re-running it skips
anything already present.

In [3]:
!pixi run -e openfold3-cuda12 setup_openfold --non-interactive 2>&1 | tail -6

Parameters already present at /home/ubuntu/.openfold3/of3-ob-2025-06-30-174k.pt
Download completed successfully.
Starting Biotite CCD setup...
Biotite CCD file at /home/ubuntu/jianwen/openfold-3/.pixi/envs/openfold3-cuda12/lib/python3.14/site-packages/biotite/structure/info/components.bcif is up-to-date with s3://openfold3-data/components.bcif, skipping.
Skipping integration tests.
Setup configuration saved to /home/ubuntu/.openfold3/setup_config.json


## 2. Inference

Predictions are driven by a query JSON. The repo ships several in
`examples/example_inference_inputs/`; ubiquitin is the smallest at 76 residues.

In [4]:
!cat examples/example_inference_inputs/query_ubiquitin.json

{
	"queries": {
		"ubiquitin": {
			"chains": [
				{
					"molecule_type": "protein",
					"chain_ids": [
						"A"
					],
					"sequence": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
				}
			]
		}
	}
}

The model needs a multiple sequence alignment. `--use-msa-server true` uses the public ColabFold
server, which is the quickest route and needs no local databases, but it does need outbound
network and is subject to their rate limits. For offline or high-volume use you would generate
alignments yourself and skip the flag.

In [5]:
!pixi run -e openfold3-cuda12 run_openfold predict \
    --query-json examples/example_inference_inputs/query_ubiquitin.json \
    --use-msa-server true \
    --output-dir ./quickstart_inference 2>&1 | tail -12

  warnings.warn(
Predicting DataLoader 0: 100%|██████████| 1/1 [00:10<00:00, 10.05s/it]

    PREDICTION SUMMARY (COMPLETE)    
Total Queries Processed: 1
  - Successful Queries:  1
  - Failed Queries:      0

Removing empty log directory...


Five diffusion samples are written per seed, each as an mmCIF structure with a matching
confidence file. `plddt` is per-atom confidence (0-100, higher is better), `pae` is the
predicted aligned error matrix, `pde` the predicted distance error.

In [6]:
!ls quickstart_inference/ubiquitin/seed_42/ && du -sh quickstart_inference

timing.json
ubiquitin_seed_42_sample_1_confidences.json
ubiquitin_seed_42_sample_1_confidences_aggregated.json
ubiquitin_seed_42_sample_1_model.cif
ubiquitin_seed_42_sample_2_confidences.json
ubiquitin_seed_42_sample_2_confidences_aggregated.json
ubiquitin_seed_42_sample_2_model.cif
ubiquitin_seed_42_sample_3_confidences.json
ubiquitin_seed_42_sample_3_confidences_aggregated.json
ubiquitin_seed_42_sample_3_model.cif
ubiquitin_seed_42_sample_4_confidences.json
ubiquitin_seed_42_sample_4_confidences_aggregated.json
ubiquitin_seed_42_sample_4_model.cif
ubiquitin_seed_42_sample_5_confidences.json
ubiquitin_seed_42_sample_5_confidences_aggregated.json
ubiquitin_seed_42_sample_5_model.cif
7.7M	quickstart_inference


In [7]:
# Analysis, not part of the workflow: mean pLDDT per sample.
import json, glob

for path in sorted(glob.glob("quickstart_inference/ubiquitin/seed_42/*_confidences.json")):
    plddt = json.load(open(path))["plddt"]
    sample = path.split("_sample_")[1].split("_")[0]
    print(f"  sample {sample}   mean pLDDT {sum(plddt) / len(plddt):6.1f}   atoms {len(plddt)}")

  sample 1   mean pLDDT   90.1   atoms 601
  sample 2   mean pLDDT   90.1   atoms 601
  sample 3   mean pLDDT   90.0   atoms 601
  sample 4   mean pLDDT   90.2   atoms 601
  sample 5   mean pLDDT   89.9   atoms 601


## 3. Training

Full training pulls a large PDB set from S3. For a smoke test the repo can instead sample a
handful of structures and download only those: 8 training and 4 validation by default, about
73 MB. It takes two commands, and the same `--train-size` has to go to both, since each
resolves its cache from that number and both default to 8.

The first run also downloads a 1.6 GB catalogue to sample from. It holds no coordinates, is
never read again once the pruned caches exist, and can be deleted afterwards.

In [8]:
!pixi run -e openfold3-cuda12 generate-pdb-subset --train-size 8 2>&1 | tail -1
!pixi run -e openfold3-cuda12 download-pdb-subset --train-size 8 2>&1 | tail -3

Wrote runner yaml to /home/ubuntu/jianwen/openfold-3/datasets/train_pdb_subset.yaml
Reference mols: 0 downloaded, 48 skipped, 0 failed

Done: 0 downloaded, 435 skipped, 1 failed


### A path trap

The repo contains `scripts/datasets/train_pdb_subset.yaml`, which looks like the config to use.
It isn't: it is a committed artifact still holding the absolute paths of whoever generated it.
The generator writes a fresh copy, with your paths, to `datasets/train_pdb_subset.yaml`.

In [9]:
!echo "committed (a template):" && grep -m1 dataset_cache_file scripts/datasets/train_pdb_subset.yaml
!echo "generated (use this):"  && grep -m1 dataset_cache_file datasets/train_pdb_subset.yaml

committed (a template):
    dataset_cache_file: /home/jnwei/workspace/openfold-3/scripts/datasets/training_cache_with_templates_subset_8.json


generated (use this):
    dataset_cache_file: /home/ubuntu/jianwen/openfold-3/datasets/training_cache_with_templates_subset_8.json


### The settings that matter

| Setting | Value | Controls |
|---|---|---|
| `pl_trainer_args.devices` | `1` | GPUs per node, the single-GPU knob |
| `pl_trainer_args.num_nodes` | `1` | Nodes |
| `pl_trainer_args.precision` | `bf16-mixed` | Mixed precision |
| `pl_trainer_args.max_epochs` | `2` | Length of the smoke test |
| `data_module_args.batch_size` | `1` | Structures per step |
| `dataset_configs...token_budget` | `384` | Crop size, the main memory lever |

One gap to close first. `LoggingConfig` accepts Weights & Biases or nothing, and the generated
config sets `wandb_config: null`, so every loss is computed and thrown away, leaving an empty
`logs/` directory beside a 6 GB checkpoint.

Enabling it takes two things, and only one is obvious. Setting `wandb_config.offline: true` in
the YAML is **not sufficient**: the flag never reaches `wandb.init`, which still tries to log in
and aborts the run before training starts. The environment variable `WANDB_MODE=offline` is what
actually works. Set both: the flag documents the intent, the variable enforces it.

In [10]:
# Add offline logging to the generated config.
import yaml

config = yaml.safe_load(open("datasets/train_pdb_subset.yaml"))
config["logging_config"] = {
    "log_lr": True,
    "wandb_config": {"project": "openfold3-quickstart", "experiment_name": "single-h100",
                     "offline": True},
}
config["experiment_settings"]["output_dir"] = "./quickstart_run"
yaml.safe_dump(config, open("datasets/quickstart_single_h100.yaml", "w"), sort_keys=False)
print("wrote datasets/quickstart_single_h100.yaml")

wrote datasets/quickstart_single_h100.yaml


Now run it. Two epochs of four steps; a few minutes, most of it validation.

In [11]:
%%bash
# Sample device memory every 2s while training runs, so the cost is measured, not claimed.
nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -lms 2000 > /tmp/of3_gpu.log 2>&1 &
SAMPLER=$!

WANDB_MODE=offline pixi run -e openfold3-cuda12 run_openfold train \
    --runner-yaml datasets/quickstart_single_h100.yaml --seed 42 2>&1 \
    | tr '\r' '\n' | grep -E "Epoch 1:|Trainer.fit. stopped" | tail -2

kill $SAMPLER 2>/dev/null
echo "peak GPU memory: $(sort -n /tmp/of3_gpu.log | tail -1) MiB"

Epoch 1: 100%|██████████| 4/4 [01:26<00:00, 21.56s/it, v_num=6bu5]`Trainer.fit` 

stopped: `max_epochs=2` reached.
Epoch 1: 100%|██████████| 4/4 [01:26<00:00, 21.

56s/it, v_num=6bu5]


peak GPU memory: 28991 MiB


### Reading the results

Everything lands under the config's `output_dir`. Note that enabling logging *moves* the
checkpoint: without wandb it is `<output_dir>/checkpoints/`, with it
`<output_dir>/<project>/<run id>/checkpoints/`. Hardcoded paths break the moment you turn
logging on.

The losses live only inside a binary `run-*.wandb` file, so the cell below converts them to a
CSV you can open with anything.

In [12]:
!find quickstart_run -maxdepth 4 \( -name "*.ckpt" -o -name "*.wandb" \) -exec du -h {} +

120K	quickstart_run/wandb/offline-run-20260924_061308-nxya6bu5/run-nxya6bu5.wandb
6.3G	quickstart_run/openfold3-quickstart/nxya6bu5/checkpoints/last.ckpt


In [13]:
# Analysis, not part of the workflow: pull the loss history out of the wandb file.
import csv, glob, json

from wandb.proto import wandb_internal_pb2 as pb
from wandb.sdk.internal import datastore

store = datastore.DataStore()
store.open_for_scan(glob.glob("quickstart_run/wandb/offline-run-*/run-*.wandb")[0])

history = []
while True:
    data = store.scan_data()
    if data is None:
        break
    record = pb.Record()
    record.ParseFromString(data)
    if record.WhichOneof("record_type") != "history":
        continue
    row = {}
    for item in record.history.item:
        key = item.key or ".".join(item.nested_key)
        try:
            row[key] = json.loads(item.value_json)
        except Exception:
            row[key] = item.value_json
    if "train/loss" in row:
        history.append(row)

columns = ["_step", "train/loss", "train/distogram_loss", "train/diffusion_loss"]
with open("quickstart_run/metrics.csv", "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(history)

print(f"{'step':>5}{'loss':>10}{'distogram':>12}{'diffusion':>12}")
print("-" * 39)
for row in history:
    line = f"{str(row.get('_step', '?')):>5}"
    for key, width in zip(columns[1:], (10, 12, 12)):
        try:
            line += f"{float(row[key]):>{width}.3f}"
        except (KeyError, TypeError, ValueError):
            line += f"{'-':>{width}}"
    print(line)
print("\nwrote quickstart_run/metrics.csv")

 step      loss   distogram   diffusion
---------------------------------------
    0     6.739       4.159       6.613
    1     6.844       4.159       6.718
    2     7.255       4.158       7.129
    3     7.242       4.157       7.116
    9     7.159       4.155       7.033
   10     7.435       4.150       7.309
   11     6.697       4.149       6.572
   12     8.066       4.139       7.941

wrote quickstart_run/metrics.csv


### What these numbers do and don't say

The loss does not fall, and should not be read as if it might. Eight optimiser steps over eight
structures is a smoke test: it proves the data pipeline, model, loss, backward pass and
checkpoint writer work together. Learning anything takes orders of magnitude more steps and
data, and a quickstart showing a tidy descending curve here would be showing you noise.

What it does establish is cost. Peak GPU memory at `token_budget: 384`, sampled during the
run above, is about 28 GiB, which fits inside
a 40 GB card, never mind an 80 GB one. Raise the token budget and that is the number that moves
first. Each run also writes a checkpoint of roughly 6 GB.

## 4. Scaling up

The same YAML covers every rung; two fields change.

| | `devices` | `num_nodes` |
|---|---|---|
| This notebook | 1 | 1 |
| One full node | 8 | 1 |
| Multi-node | 8 | 32 |

The launch command does not change. Real training then swaps the sampled subset for the full
PDB set:

```bash
aws s3 sync s3://openfold3-data/pdb_training_set/ /shared/openfold3/pdb_training_set/ --no-sign-request
```

and starts from a config in `examples/training_yamls/` (`initial_training.yml` through
`finetune_3.yml`), which carry real schedules rather than a two-epoch smoke test.

## Links

- OpenFold3: <https://github.com/aqlaboratory/openfold-3>
- Training guide: <https://openfold-3.readthedocs.io/en/latest/training.html>
- Configuration reference: <https://openfold-3.readthedocs.io/en/latest/configuration_reference.html>
- Datasets: <https://portal.openfold.omsf.io/datasets>